# 4.6. Выручка и потери от отмен

## Revenue analysis: GMV, потери от отмен, AOV
### 1. Загрузка данных
### 2. Парсинг product_ids (массив → строки)
### 3. Сумма заказа (order_amount)
### 4. Флаг отмены
### 5. Метрики: GMV, lost revenue, AOV
### 6. Выводы

In [5]:
import pandas as pd
import ast

# ---------- 1. Загрузка ----------
orders   = pd.read_csv('orders.csv')
products = pd.read_csv('products.csv')
actions  = pd.read_csv('user_actions.csv')

# ---------- 2. Парсинг product_ids ----------
# '[65,28]' -> [65, 28]
orders['product_ids'] = orders['product_ids'].apply(ast.literal_eval)

# разворачиваем: одна строка на позицию
orders_exploded = orders.explode('product_ids').rename(
    columns={'product_ids': 'product_id'}
)
orders_exploded['product_id'] = orders_exploded['product_id'].astype(int)

# ---------- 3. Цены и сумма заказа ----------
order_items = orders_exploded.merge(products, on='product_id', how='left')

# проверка: сколько позиций без цены
missing = order_items['price'].isna().sum()
print(f"Позиций без цены: {missing}")

order_amount = (order_items
                .groupby('order_id')['price']
                .sum()
                .reset_index(name='order_amount'))

# ---------- 4. Флаг отмены ----------
cancels = set(
    actions.loc[actions['action'] == 'cancel_order', 'order_id']
)
order_amount['is_canceled'] = order_amount['order_id'].isin(cancels)

# ---------- 5. Метрики ----------
gmv           = order_amount['order_amount'].sum()
lost_revenue  = order_amount.loc[order_amount['is_canceled'], 'order_amount'].sum()
success_rev   = order_amount.loc[~order_amount['is_canceled'], 'order_amount'].sum()

aov_success   = order_amount.loc[~order_amount['is_canceled'], 'order_amount'].mean()
aov_canceled  = order_amount.loc[order_amount['is_canceled'], 'order_amount'].mean()

lost_pct      = lost_revenue / gmv * 100

print(f"GMV: {gmv:,.0f}")
print(f"Lost revenue: {lost_revenue:,.0f} ({lost_pct:.2f}% от GMV)")
print(f"AOV успешных: {aov_success:,.0f}")
print(f"AOV отменённых: {aov_canceled:,.0f}")

Позиций без цены: 0
GMV: 22,821,728
Lost revenue: 1,142,633 (5.01% от GMV)
AOV успешных: 383
AOV отменённых: 384
